VCT Match Predictor:

Takes data from vlr.gg of official Valorant VCT Matches and aims to use machine learning to predict outcomes of matches

In [11]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


In [12]:
# Load and clean data
df = pd.read_csv("../Data/team_stats3.csv")
with open('../Data/Dictionaries/team_names.txt', encoding='utf-8') as f:
    valid_teams = set(line.strip() for line in f if line.strip())

df = df[df['team'].isin(valid_teams) & df['opponent'].isin(valid_teams)]
df = df.dropna(subset=['total_kills','total_deaths','total_assists','avg_acs'])
df = df.drop(['series_score', 'map_url', 'avg_kast'], axis=1, errors='ignore')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df['row_id'] = df.index  # Unique row id for strict ordering

# Remove one side of mirrored matches
df_clean = df.copy()
to_remove = []

for idx, row in df.iterrows():
    if idx in to_remove:
        continue
        
    # Find mirror match
    mirror = df[
        (df['date'] == row['date']) & 
        (df['team'] == row['opponent']) & 
        (df['opponent'] == row['team'])
    ]
    
    if len(mirror) > 0:
        # Keep the one with lower index, remove the other
        mirror_idx = mirror.index[0]
        if mirror_idx > idx:
            to_remove.append(mirror_idx)

print(f"Removing {len(to_remove)} duplicate matches")
df_clean = df_clean.drop(to_remove).reset_index(drop=True)


Removing 1616 duplicate matches


Clean the dataframe:
Remove unofficial games (games where the opponent is not in VCT)
Remove empty games (games with missing data in kills, deaths, assists)
Remove unwanted columns (series_score [incorrect data], map_url)

In [13]:
# Rolling win rate calculation with temporal filtering
def rolling_winrate(df, team, match_date, window):
    history = df[(df['team'] == team) & (df['date'] < match_date)]
    lastn = history.tail(window)
    if lastn.empty:
        return np.nan
    return lastn['won_map'].mean()

for w in [3, 5, 10]:
    df_clean[f'team_winrate_last{w}'] = np.nan
    df_clean[f'opp_winrate_last{w}'] = np.nan

for idx, row in df_clean.iterrows():
    match_date = row['date']
    team = row['team']
    opponent = row['opponent']
    for w in [3, 5, 10]:
        df_clean.at[idx, f'team_winrate_last{w}'] = rolling_winrate(df_clean, team, match_date, w)
        df_clean.at[idx, f'opp_winrate_last{w}'] = rolling_winrate(df_clean, opponent, match_date, w)

# Calculate rolling averages for stats with temporal filtering
for stat in ['total_kills', 'total_deaths', 'total_assists', 'avg_acs']:
    for w in [3, 5, 10]:
        df_clean[f'team_{stat}_last{w}'] = np.nan
        df_clean[f'opp_{stat}_last{w}'] = np.nan

for idx, row in df_clean.iterrows():
    match_date = row['date']
    team = row['team']
    opponent = row['opponent']
    for stat in ['total_kills', 'total_deaths', 'total_assists', 'avg_acs']:
        for w in [3, 5, 10]:
            # Use date filtering instead of row_id
            team_hist = df_clean[(df_clean['team'] == team) & (df_clean['date'] < match_date)].sort_values('date', ascending=False).head(w)
            opp_hist = df_clean[(df_clean['team'] == opponent) & (df_clean['date'] < match_date)].sort_values('date', ascending=False).head(w)
            df_clean.at[idx, f'team_{stat}_last{w}'] = team_hist[stat].mean() if not team_hist.empty else np.nan
            df_clean.at[idx, f'opp_{stat}_last{w}'] = opp_hist[stat].mean() if not opp_hist.empty else np.nan

# Head-to-head stats with temporal filtering
def h2h_stats(df, team, opponent, match_date):
    prev_matches = df[
        (((df['team'] == team) & (df['opponent'] == opponent)) |
         ((df['team'] == opponent) & (df['opponent'] == team)))
        & (df['date'] < match_date)  # Use date instead of row_id
    ]
    team_wins = prev_matches[(prev_matches['team'] == team) & (prev_matches['won_map'] == True)].shape[0]
    total_matches = prev_matches.shape[0]
    winrate = team_wins / total_matches if total_matches > 0 else np.nan
    return team_wins, total_matches, winrate

df_clean['h2h_team_wins'] = np.nan
df_clean['h2h_total_matches'] = np.nan
df_clean['h2h_winrate'] = np.nan
df_clean['opp_h2h_team_wins'] = np.nan
df_clean['opp_h2h_total_matches'] = np.nan
df_clean['opp_h2h_winrate'] = np.nan

for idx, row in df_clean.iterrows():
    match_date = row['date']
    team = row['team']
    opponent = row['opponent']
    team_wins, total_matches, winrate = h2h_stats(df_clean, team, opponent, match_date)
    opp_wins, opp_total_matches, opp_winrate = h2h_stats(df_clean, opponent, team, match_date)
    df_clean.at[idx, 'h2h_team_wins'] = team_wins
    df_clean.at[idx, 'h2h_total_matches'] = total_matches
    df_clean.at[idx, 'h2h_winrate'] = winrate
    df_clean.at[idx, 'opp_h2h_team_wins'] = opp_wins
    df_clean.at[idx, 'opp_h2h_total_matches'] = opp_total_matches
    df_clean.at[idx, 'opp_h2h_winrate'] = opp_winrate


In [14]:
# Update feature selection to use only rolling stats and engineered features
exclude_cols = [
    'date', 'map_url', 'series_score', 'team', 'opponent', 'patch', 'map',
    'agent1', 'agent2', 'agent3', 'agent4', 'agent5', 'won_map', 'won_series', 'row_id',
    'total_kills', 'total_deaths', 'total_assists', 'avg_acs'
]
feature_cols = [col for col in df_clean.columns if col not in exclude_cols and pd.api.types.is_numeric_dtype(df_clean[col])]


In [15]:
# Model training
X = df_clean[feature_cols]
y = df_clean['won_map']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.35, random_state=42)

clf = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_pred))

Test accuracy: 0.997557251908397


/Users/bitanga/Coding/vct-predictions-main/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [13:16:31] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
